![Week 6 Day 4](../assets/banners/w6d4.png)

# Week 6 Day 4 - One Command to Run It All: Docker Compose and Deployment

GradeWise now has an API (Day 1), a dashboard with a review queue (Day 2), and live progress, report cards and chat (Day 3). To run it, you open two terminals and type two commands. On *your* machine, with *your* Python, *your* `.venv`, *your* Windows quirks.

Now imagine handing it to the IT team at the Northbridge Institute. "Install uv, then Python 3.12, then…" No. Today we package GradeWise so it runs **the same way everywhere**, with one command: `docker compose up`. Then we put it on the internet.

You may not have Docker installed yet (this notebook doesn't need it), so today's Docker commands are shown as markdown, ready to copy. The cells that *can* run offline do: we read the Dockerfile and compose file line by line, simulate the build cache, check what `.dockerignore` keeps out, and run the container's real health check against a real API server.

![Week 6 roadmap](../assets/roadmap/week6.png)

### 🎯 By the end of this lab you will
1. 🐳 Know the four Docker words that matter: image, container, volume, compose
2. 📜 Read GradeWise's `Dockerfile` line by line, and know why the order of the lines makes builds fast
3. 🙈 Check that secrets and junk never reach the image (`.dockerignore`)
4. 🧩 Read `docker-compose.yml`: two services, one image, one volume, a health-gated start
5. 🚀 Compare three ways to deploy (Hugging Face Spaces, Render, Fly.io) plus LangGraph Platform, and run a pre-flight checklist

### 🎬 Videos for this lab

| # | Lecture | Type |
|:-:|---|:-:|
| 1 | Day 4 - One Command to Run It All: Docker Compose | 🧪 |
| 2 | Day 4 - Deploying GradeWise (Render / Fly.io / HF Spaces) | 🧪 |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Install Docker Desktop (Windows / Mac) or Docker Engine (Linux) from <a href="https://docs.docker.com/get-docker/">docs.docker.com/get-docker</a> to run today's commands yourself. On Windows, Docker Desktop uses WSL 2; the installer sets it up. <b>Everything in this notebook runs without Docker</b>, and deploying costs nothing until you choose a paid plan.</span>
        </td>
    </tr>
</table>

💡 **Fast & cheap run.** Already know the ideas and just want to watch everything run? Put `DATA_SIZE=small` in your `.env` (2 PDFs · Q1–Q3 · 2 students) and keep the default `MODEL_MODE=cheap` (`openai/gpt-6-luna` via OpenRouter, a few cents of API calls). Or set `MODEL_MODE=free` to run on OpenRouter's free models for $0, with the same `OPENROUTER_API_KEY` and no credits: it's rate-limited to 20 requests a minute and 50 a day (1,000 a day once you've ever bought $10 of credits), so pair it with `DATA_SIZE=small`; the model changes from call to call, so results vary; and free providers may log or train on prompts, which is fine for this course's fictional data and never for real students' work. Restart the kernel after editing `.env`; the setup cell below prints what you're running on.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

from gradewise import datasets
from gradewise.config import settings

load_dotenv(override=True)

# 💡 MODEL_MODE=cheap (default) and MODEL_MODE=free both need just OPENROUTER_API_KEY
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"ℹ️ No {key_name} set. That's fine this week: every cell runs offline in 🧪 demo mode.")
        print("   (To grade for real later, see Part 5 of the setup guide, or setup/SWAP_PROVIDER.md)")

# 📦 What this run uses: MODEL_MODE and DATA_SIZE in .env (restart the kernel after changing them)
print(settings.describe_models())
print(datasets.describe())

In [ ]:
import shutil
import subprocess
from pathlib import Path

from gradewise.config import REPO_ROOT

docker = shutil.which("docker")
if docker:
    version = subprocess.run([docker, "--version"], capture_output=True, text=True).stdout.strip()
    print(f"🐳 Docker found: {version}. You can run today's commands for real!")
else:
    print("ℹ️ Docker isn't installed here. No problem: read along today, and run the commands once it is.")

## Part 1: Why containers? Four words

"It works on my machine" is the oldest bug in software. GradeWise depends on Python 3.12, about 150 packages pinned in `uv.lock`, native libraries (PyMuPDF, Chroma's SQLite), and a folder layout. A **container** ships all of that together.

| Word | What it is | GradeWise example |
|---|---|---|
| **Image** | A read-only snapshot: an OS, Python, your packages and your code, built from a `Dockerfile` | `gradewise:latest` |
| **Container** | A running process started from an image, with its own filesystem and network | the `api` and the `ui` |
| **Volume** | Storage that lives **outside** the container, so data survives restarts and upgrades | `output` → `/app/output` |
| **Compose** | A YAML file that starts several containers together and wires up their network and volumes | `docker-compose.yml` |

The rule that keeps you sane: **containers are disposable, volumes are not.** You can delete and rebuild the `api` container ten times a day; the exams, report cards and checkpoints live in the volume.

## Part 2: The Dockerfile, line by line

Open `docker/Dockerfile`. Here it is, without the comment lines, with line numbers:

In [ ]:
dockerfile = (REPO_ROOT / "docker" / "Dockerfile").read_text(encoding="utf-8").splitlines()
for number, line in enumerate(dockerfile, start=1):
    if line.strip() and not line.lstrip().startswith("#"):
        print(f"{number:>3} │ {line}")

| Instruction | What it does | Why it's written this way |
|---|---|---|
| `FROM python:3.12-slim` | Start from Debian with Python 3.12 | `slim` is a few hundred MB smaller than the full image; same Python as `.python-version` |
| `COPY --from=ghcr.io/astral-sh/uv:0.12.23 /uv /uvx …` | Copy the `uv` binary out of its official image | **Pinned version**: builds must be repeatable. No `pip install uv` |
| `ENV PYTHONUNBUFFERED=1 …` | Settings for every later step and for the running app | Unbuffered logs appear instantly in `docker logs`; `UV_PROJECT_ENVIRONMENT=/app/.venv` + `PATH` make `python` mean the venv; `API_HOST=0.0.0.0` (see below) |
| `WORKDIR /app` | `cd /app` for everything after | |
| `COPY pyproject.toml uv.lock README.md ./` then `uv sync --frozen --no-dev --no-install-project` | Install **only the dependencies** | 🧱 **Layer caching**: this slow layer is reused until the lock file changes. `--frozen` installs exactly `uv.lock`; `--no-dev` skips pytest, ruff and nbconvert |
| `RUN --mount=type=cache,target=/root/.cache/uv` | A build cache for downloaded wheels | Rebuilds don't re-download LangChain and Chroma |
| `COPY gradewise ./gradewise` + `COPY data ./data` + `uv sync` | The code and the course data | Changes often, so it comes **after** the dependencies |
| `RUN useradd … && chown …` + `USER gradewise` | Create and switch to an unprivileged user | 🔒 If anything is ever exploited, it owns one user, not root |
| `VOLUME ["/app/output"]` | Declare the data folder as a volume | Databases, vectors, uploads and report cards outlive the container |
| `EXPOSE 8000 8501` | Document the ports | API and dashboard |
| `HEALTHCHECK … /health` | Docker asks "are you really serving?" every 30 s | `python` instead of `curl`: the slim image has no curl |
| `CMD ["python", "-m", "gradewise.api"]` | What runs by default | ⚠️ **One** worker: the batch registry lives in memory (Day 1) |

💡 **Why `API_HOST=0.0.0.0`?** On your laptop the API listens on `127.0.0.1`, which means "only this machine". Inside a container, "this machine" is the container itself, so nothing outside (not even the dashboard container) could reach it. `0.0.0.0` means "every network interface".

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">You change one line in <code>gradewise/api/main.py</code> and run <code>docker compose build</code> again. Which steps re-run, and which come straight from the cache? Then: what if you <code>uv add</code> a new package instead? Predict both, then run the next cell.</span>
        </td>
    </tr>
</table>

In [ ]:
# 💡 A toy model of Docker's layer cache: a step re-runs if its own inputs changed OR any step before it re-ran.
STEPS = [
    ("FROM python:3.12-slim", set()),
    ("COPY uv binary", set()),
    ("COPY pyproject.toml uv.lock README.md", {"pyproject.toml", "uv.lock", "README.md"}),
    ("RUN uv sync --no-install-project (deps: slow!)", set()),
    ("COPY gradewise", {"gradewise/"}),
    ("COPY data", {"data/"}),
    ("RUN uv sync (install the project)", set()),
    ("RUN useradd + chown", set()),
]


def build(changed: set[str]) -> None:
    dirty = False
    for name, inputs in STEPS:
        dirty = dirty or any(c == i or c.startswith(i) for c in changed for i in inputs)
        print(f"   {'🔨 RUN  ' if dirty else '⚡ cache'} {name}")


print("You edited gradewise/api/main.py:")
build({"gradewise/api/main.py"})
print("\nYou added a package (pyproject.toml + uv.lock changed):")
build({"pyproject.toml", "uv.lock"})

Editing code rebuilds in seconds, because the expensive dependency layer comes from the cache. Put `COPY . .` at the top instead and **every** code change re-installs every package. Ordering a Dockerfile from "changes rarely" to "changes often" is the single most useful Docker habit.

## Part 3: `.dockerignore`: what never goes in

`docker build` first sends a **build context** (the folder) to the builder. `.dockerignore` filters it, which keeps builds fast and, far more importantly, keeps **secrets** out of the image. Anything copied into an image layer can be read by anyone who gets the image, *even if a later step deletes it*.

Let's check a few paths with a simplified version of Docker's matching rules (the last matching pattern wins; `!` re-includes):

In [ ]:
from fnmatch import fnmatch

rules = [
    line.strip()
    for line in (REPO_ROOT / ".dockerignore").read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.strip().startswith("#")
]


def ignored(rel: str) -> bool:
    """Simplified .dockerignore matching: last matching rule wins, '!' re-includes, 'dir/' covers its contents."""
    result = False
    for rule in rules:
        negate, pattern = rule.startswith("!"), rule.lstrip("!").rstrip("/")
        candidates = [pattern, pattern.removeprefix("**/")]
        hit = any(fnmatch(rel, p) or rel.startswith(p + "/") or fnmatch(Path(rel).name, p) for p in candidates)
        if hit:
            result = not negate
    return result


for rel in [
    ".env",
    ".env.example",
    ".venv/Lib/site-packages/x.py",
    "output/gradewise.db",
    "week6/day4.ipynb",
    "tests/test_api.py",
    "uv.lock",
    "gradewise/api/main.py",
    "data/knowledge-base/unit1_operating_systems.pdf",
]:
    print(f"{'🙈 excluded' if ignored(rel) else '📦 sent    '}  {rel}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;"><b>Your API key must never be in an image.</b> <code>.env</code> is excluded by <code>.dockerignore</code> (and by <code>.gitignore</code>), and keys reach the containers at <b>run time</b> instead: through <code>env_file: .env</code> in compose on your machine, and through each host's "secrets" screen when you deploy. If a key ever does end up in an image or a commit, <b>revoke it</b> in your provider's dashboard: deleting the file afterwards doesn't remove it from history.</span>
        </td>
    </tr>
</table>

## Part 4: `docker-compose.yml`: one command, two services

Open `docker-compose.yml`. Here it is without comments:

In [ ]:
compose_text = (REPO_ROOT / "docker-compose.yml").read_text(encoding="utf-8")
for line in compose_text.splitlines():
    code = line.split(" #")[0].rstrip()
    if code and not code.lstrip().startswith("#"):
        print(code)

YAML anchors (`&gradewise`) and merge keys (`<<: *gradewise`) let both services share the build, image, `.env`, volume and restart policy. Let's load it the way Compose does and see what each service really gets:

In [ ]:
import yaml

compose = yaml.safe_load(compose_text)  # 💡 safe_load understands anchors and << merges
for name, svc in compose["services"].items():
    command = " ".join(map(str, svc.get("command", ["(image default)"])))
    print(f"🧩 {name}")
    print(f"   command    : {command[:90]}")
    print(f"   ports      : {svc.get('ports', [])}")
    print(f"   volumes    : {svc.get('volumes', [])}")
    print(f"   environment: {svc.get('environment', {})}")
    if "depends_on" in svc:
        print(f"   depends_on : {svc['depends_on']}")
    if "profiles" in svc:
        print(f"   profiles   : {svc['profiles']}  (only starts when asked)")
print("\nnamed volumes:", list(compose["volumes"]))

Five things to notice:

1. **One image, two commands.** `api` runs `python -m gradewise.api`; `ui` runs Streamlit. Same code, same lock file, built once.
2. **Service names are hostnames.** On Compose's private network, the dashboard reaches the API at `http://api:8000`, which is why `API_URL` (the variable `gradewise/ui/client.py` reads) is set to exactly that.
3. **One shared volume**, `output`, mounted at `/app/output`. Run `docker compose down` and your report cards are still there next time.
4. **Health-gated start.** `depends_on: api: condition: service_healthy` waits for the API's `HEALTHCHECK` to pass, not just for its process to exist, so the dashboard never opens to an "API offline" page.
5. **Profiles.** `ollama` only starts with `--profile ollama`: a free local model server for people without an API key.

### The commands

```bash
docker compose up --build          # build the image, start api + ui → http://localhost:8501
docker compose up -d               # the same, in the background ("detached")
docker compose logs -f api         # follow the API's logs
docker compose ps                  # what's running, and is it healthy?
docker compose exec api python -m gradewise info   # run a command inside the api container
docker compose down                # stop and remove the containers (the volume stays)
docker compose down -v             # ⚠️ ...and DELETE the volume: every exam, report and checkpoint
docker compose --profile ollama up --build          # + a local Ollama model server
```

Free demo mode, no key needed:

```bash
GRADEWISE_DEMO=true docker compose up --build             # Mac / Linux
$env:GRADEWISE_DEMO="true"; docker compose up --build     # Windows PowerShell
```

## Part 5: The health check, for real

Docker, Compose, Render and Fly all decide whether GradeWise is alive the same way: they run the `HEALTHCHECK` command and look at its exit code. We can't start a container here, but we *can* start a real API server and run the Dockerfile's exact health-check code against it:

In [ ]:
import json
import shutil
import time

from fastapi.testclient import TestClient

from gradewise.api.main import create_app
from gradewise.config import OUTPUT_DIR, REPO_ROOT
from gradewise.console import success

SANDBOX = OUTPUT_DIR / "week6" / "day4"
shutil.rmtree(SANDBOX, ignore_errors=True)  # 💡 a fresh start every time you run the notebook

app = create_app(SANDBOX, demo=True)  # 🧪 demo=True: offline fakes, no API key, no cost
client = TestClient(app)  # 💡 calls the app in-process: real HTTP semantics, no server, no port


def show(data, limit: int = 1500) -> None:
    """Pretty-print JSON, cut short so a big response doesn't flood the notebook."""
    text = json.dumps(data, indent=2, ensure_ascii=False, default=str)
    print(text if len(text) <= limit else text[:limit] + "\n  …")


def wait_for(batch_id: str, *states: str, timeout: float = 120) -> dict:
    """Poll GET /batches/{id} until the batch reaches one of `states` (default: paused, done or failed)."""
    states = states or ("paused", "done", "failed")
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        batch = client.get(f"/batches/{batch_id}").json()
        if batch["status"] in states:
            return batch
        time.sleep(0.2)
    raise TimeoutError(f"{batch_id} never reached {states}")


success("A private GradeWise API for this notebook", {"Mode": "🧪 demo (offline fakes)", "Writes to": str(SANDBOX)})

In [ ]:
import logging
import socket
import sys
import threading

import uvicorn


def start_api(app, port: int | None = None) -> tuple[str, uvicorn.Server]:
    """Serve `app` with uvicorn on a background thread, on 127.0.0.1 only. Returns (url, server)."""
    if port is None:
        with socket.socket() as s:  # 💡 ask the OS for a free port, so nothing clashes
            s.bind(("127.0.0.1", 0))
            port = s.getsockname()[1]
    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning", access_log=False))
    threading.Thread(target=server.run, daemon=True).start()
    while not server.started:
        time.sleep(0.05)
    return f"http://127.0.0.1:{port}", server


for name in ("uvicorn", "uvicorn.access", "uvicorn.error", "httpx", "streamlit"):
    logging.getLogger(name).setLevel(logging.ERROR)  # 💡 keep the notebook quiet

API_URL, server = start_api(app)
port = API_URL.rsplit(":", 1)[1]

# 💡 Pull the python one-liner out of the Dockerfile's HEALTHCHECK, so we test the real thing.
source = "\n".join(dockerfile)
check = source.split("HEALTHCHECK", 1)[1].split('CMD python -c "', 1)[1].split('" || exit 1', 1)[0]
check = check.replace('\\"', '"')
print("HEALTHCHECK runs:", check, "\n")

for label, env_port in [("API running", port), ("wrong port ", "9")]:
    result = subprocess.run(
        [sys.executable, "-c", check],
        env={**os.environ, "API_PORT": env_port},
        capture_output=True,
        text=True,
        timeout=30,
    )
    print(f"{label}: exit code {result.returncode} → {'healthy ✅' if result.returncode == 0 else 'unhealthy ❌'}")
server.should_exit = True

Exit code `0` means healthy; anything else, three times in a row, and Docker marks the container **unhealthy** (and Compose, Render or Fly can restart it).

## Part 6: Deploying GradeWise

Read `week6/project/DEPLOY.md`: it walks through every option step by step. Here's how to choose:

| | 🤗 Hugging Face Spaces | 🟣 Render | 🎈 Fly.io | 🦜 LangGraph Platform |
|---|---|---|---|---|
| Best for | A free public **portfolio demo** | A simple always-on app | Production-shaped, close to users | Hosting the **graphs**, not the app |
| Shape | One container: API + UI together, UI on port 7860 | Two web services + a disk | Two processes + a volume, private network | Your graphs behind LangGraph's API |
| Persistence | Wiped on restart (free tier) | Persistent disk (paid instance) | Volumes | Managed Postgres |
| Rough cost | Free (CPU basic) | ≈ $7–8/month + disk | ≈ $5–10/month, pay as you go | Free dev tier, then usage-based |
| Effort | ⭐ | ⭐⭐ | ⭐⭐⭐ | ⭐⭐ (new API shape) |

Prices change: always check the host's pricing page.

### 🤗 The five-minute portfolio demo (Hugging Face Spaces, Docker SDK)

```dockerfile
# Dockerfile at the root of your Space (the same lines as docker/Dockerfile, up to USER gradewise), then:
ENV GRADEWISE_DEMO=true API_URL=http://127.0.0.1:8000
EXPOSE 7860
CMD ["sh", "-c", "python -m gradewise.api & python -m streamlit run gradewise/ui/app.py --server.address=0.0.0.0 --server.port=7860"]
```
and in the Space's `README.md` front matter: `sdk: docker` and `app_port: 7860`. Push, and the build log turns into a live URL.

### 🟣 Render (two services)
```text
gradewise-api : Docker, docker/Dockerfile, env API_PORT=10000 + your secrets, disk at /app/output, health check /health
gradewise-ui  : same image, command "python -m streamlit run gradewise/ui/app.py --server.address=0.0.0.0 --server.port=10000",
                env API_URL=https://gradewise-api.onrender.com, health check /_stcore/health
```

### 🎈 Fly.io
```bash
fly launch --dockerfile docker/Dockerfile --no-deploy     # writes fly.toml: add [processes] api + ui, a [[mounts]]
fly volumes create gradewise_output --size 1
fly secrets set GRADEWISE_DEMO=true                      # or: fly secrets set OPENROUTER_API_KEY=sk-or-...
fly deploy && fly open
```

### 🦜 Or let LangGraph Platform host the graphs
A `langgraph.json` points at `gradewise.orchestrator` and `gradewise.exam.graph`, and the platform provides threads, persistent checkpoints, background runs, interrupts and streaming as a managed API: most of what we built by hand this week. The trade-off: less control over the API's shape, and a vendor in the loop. DEPLOY.md has the details.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;"><b>A public link with your API key behind it is a public credit card.</b> Deploy public demos with <code>GRADEWISE_DEMO=true</code>. If you deploy with real models: put a login in front (your host's access control, an OAuth proxy, or a bearer-token dependency in FastAPI), set a hard monthly spending cap in your provider's billing settings <i>before</i> you share the URL, and never upload real students' work to a public demo.</span>
        </td>
    </tr>
</table>

## Part 7: A pre-flight checklist

Before you deploy anything, run a checklist. Here's one that checks the repo's deployment files offline (all of these should be ✅ in the course repo; keep them that way in your fork):

In [ ]:
dockerignore = (REPO_ROOT / ".dockerignore").read_text(encoding="utf-8")
gitignore = (REPO_ROOT / ".gitignore").read_text(encoding="utf-8")
checks = {
    ".env is excluded from the image": ignored(".env"),
    ".env is excluded from git": any(line.strip() in (".env", "/.env") for line in gitignore.splitlines()),
    "uv.lock exists (repeatable installs)": (REPO_ROOT / "uv.lock").exists(),
    "dependencies installed --frozen": "--frozen" in source,
    "runs as a non-root USER": any(line.startswith("USER ") for line in dockerfile),
    "has a HEALTHCHECK": "HEALTHCHECK" in source,
    "output/ is a volume": "/app/output" in source and "output" in compose["volumes"],
    "UI waits for a healthy API": compose["services"]["ui"]["depends_on"]["api"]["condition"] == "service_healthy",
    "UI finds the API by service name": compose["services"]["ui"]["environment"]["API_URL"] == "http://api:8000",
}
for check, ok in checks.items():
    print(f"{'✅' if ok else '❌'} {check}")
print(f"\n{sum(checks.values())}/{len(checks)} checks passed")

And the checklist a computer can't run for you (from DEPLOY.md's hardening section):

- [ ] A login in front of anything with real keys or real students' work
- [ ] HTTPS only (all three hosts do this for you)
- [ ] Upload size limits (`server.maxUploadSize` for the UI; a body-size limit for the API)
- [ ] Backups of the volume, at least `gradewise.db`
- [ ] LangSmith tracing on, so any mark can be explained after the fact
- [ ] A retention policy: delete uploads and reports at the end of term (more on this tomorrow)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">"One command to run it" is what turns a project into something a school's IT team will even consider. Schools and exam boards often <b>must</b> keep student data on their own servers or in their own region: a Docker image they can run on-premises, with keys and data they control, removes the biggest objection to an AI grading tool before it's raised. It's also how you'd offer a hosted SaaS and a self-hosted enterprise edition from the same code.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">Today's files: <code>docker/Dockerfile</code> (one image for both services), <code>docker-compose.yml</code> (API + dashboard + optional Ollama, one volume), <code>.dockerignore</code> (secrets and junk stay out), and <code>week6/project/DEPLOY.md</code> (Hugging Face Spaces, Render, Fly.io, LangGraph Platform, costs, hardening, scaling). <code>week6/project/README.md</code> has the whole app on one page. The milestone command: <code>docker compose up --build</code>, then open <a href="http://localhost:8501">localhost:8501</a>.</span>
        </td>
    </tr>
</table>

## Recap, and where we are heading

Today you:
- 🐳 learned image, container, volume and compose, and the rule "containers are disposable, volumes are not"
- 📜 read the Dockerfile line by line, and simulated why "dependencies first, code last" makes rebuilds fast
- 🙈 checked that `.env` and other junk never reach the image
- 🧩 read the compose file: one image, two services, a shared volume, service-name networking and a health-gated start
- 🩺 ran the container's real health check against a real API server
- 🚀 compared Hugging Face Spaces, Render, Fly.io and LangGraph Platform, and ran a pre-flight checklist

**Tomorrow (Day 5):** the last day of the course. We talk honestly about **responsible AI grading**, run the whole capstone end to end in one cell, look at where to take GradeWise next, turn it into a portfolio piece, and say thank you.

⏭ **Next:** `week6/day5.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">Make deployment repeatable. Write a <b>Render Blueprint</b> (<code>render.yaml</code>) that describes both services from DEPLOY.md: <code>gradewise-api</code> (Docker, <code>docker/Dockerfile</code>, a 1 GB disk at <code>/app/output</code>, health check <code>/health</code>, <code>GRADEWISE_DEMO=true</code>) and <code>gradewise-ui</code> (same image, the Streamlit command, <code>API_URL</code> pointing at the API). Write it from Python into <code>output/week6/</code>, load it back with <code>yaml.safe_load</code>, and write a few asserts that would catch a broken deploy (no disk, wrong port, a secret value committed in the file). Solution: <code>week6/solutions/day4_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• Docker: get started: <a href="https://docs.docker.com/get-started/">docs.docker.com/get-started</a><br/>• Using uv in Docker (the official guide our Dockerfile follows): <a href="https://docs.astral.sh/uv/guides/integration/docker/">docs.astral.sh/uv/guides/integration/docker</a><br/>• Compose file reference: <a href="https://docs.docker.com/reference/compose-file/">docs.docker.com/reference/compose-file</a><br/>• Hugging Face Docker Spaces: <a href="https://huggingface.co/docs/hub/spaces-sdks-docker">huggingface.co/docs/hub/spaces-sdks-docker</a> · Render: <a href="https://render.com/docs">render.com/docs</a> · Fly.io: <a href="https://fly.io/docs/">fly.io/docs</a><br/>• LangGraph Platform: <a href="https://docs.langchain.com/langgraph-platform">docs.langchain.com/langgraph-platform</a></span>
        </td>
    </tr>
</table>